In [2]:
import gzip
import json
import requests
import io
import pandas as pd

In [3]:
# Example URL for an hourly archive (January 1, 2026 at 3 PM UTC)
url = "https://data.gharchive.org/2026-01-01-12.json.gz"

In [4]:
print(f"Downloading data from {url}...")
response = requests.get(url, stream=True, timeout=30)
response.raise_for_status()

In [4]:
print(response.content[:100])  # Print the first 100 bytes of the response content

b'\x1f\x8b\x08\x08\xfcpVi\x02\x032026-01-01-12.json\x00\xec\\\xebw\xd4F\x96\xff\xbe\x7f\x85\xe2\x9c\xd9\x04\xaf\xdb-\xa9\xf5h\xf90\xcc\x00&\xc4\xbb\x01\xbc\xc4IvC\xb2vI\xaa\xeeV\xac\x96:z\xd8\xe90\x9cC\xdb\x03\x01\x02y\xce\x92\xd72@`0\t3N2Iv\x93\t\x84?'


In [5]:
# Decompress and read line by line (since it's newline-delimited JSON)
event_count = 0
push_events = 0

In [ ]:
# Decompressing in-memory
with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
  for line in f:
    event_count += 1
    event = json.loads(line.decode("utf-8"))
    print(event)  # Print the entire event for inspection
    # Example: Filter for PushEvents
    if event.get("type") == "PushEvent":
      push_events += 1
      repo_name = event["repo"]["name"]
      actor = event["actor"]["login"]
      # Here you could write/stream this record to PostgreSQL, DuckDB, or Parquet files

    # Just inspect the first few events
    if event_count <= 3:
      print(
          f"\nEvent Type: {event.get('type')}, Repo: {event['repo']['name']}"
      )

print(f"\nProcessed {event_count} events total.")
print(f"Found {push_events} PushEvents in this hour.")


Event Type: PushEvent, Repo: dasepsamsul/Data-Analysis

Event Type: PushEvent, Repo: gschubert05/arb-arena

Event Type: DeleteEvent, Repo: aws-aemilia-kix/Github-PR-Commit-Integration-Test-DoNotTouch-Gith-RB7P4TGB-v2-preprod-ap-northeast-3

Processed 154722 events total.
Found 115695 PushEvents in this hour.


In [5]:
events_data = []
# max_rows = 1000  # Limit to first 1000 rows for quick inspection

with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
  for i, line in enumerate(f):
    # if i >= max_rows:
    #   break
    event = json.loads(line.decode("utf-8"))

    # Flatten top-level fields and key nested fields for a clean table
    flat_event = {
        "id": event.get("id"),
        "type": event.get("type"),
        "created_at": event.get("created_at"),
        "actor": event.get("actor", {}),
        # "actor_login": event.get("actor", {}).get("login"),
        "repo": event.get("repo", {}),
        "payload": event.get("payload", {}),
        "public": event.get("public"),
    }
    events_data.append(flat_event)

# Create the DataFrame
df = pd.DataFrame(events_data)


In [8]:

# Display basic info about columns and data types
print("\n--- DataFrame Info ---")
print(df.info())




--- DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id          154722 non-null  str   
 1   type        154722 non-null  str   
 2   created_at  154722 non-null  str   
 3   actor       154722 non-null  object
 4   repo        154722 non-null  object
 5   payload     154722 non-null  object
 6   public      154722 non-null  bool  
dtypes: bool(1), object(3), str(3)
memory usage: 7.2+ MB
None


In [10]:
# Display the first 5 rows
# print("\n--- First 5 Rows ---")
print(df.head())


           id               type            created_at  \
0  7118646641          PushEvent  2026-01-01T12:00:00Z   
1  7118646642          PushEvent  2026-01-01T12:00:00Z   
2  7118646643        DeleteEvent  2026-01-01T12:00:00Z   
3  5566455635  IssueCommentEvent  2026-01-01T12:00:01Z   
4  5566455640   PullRequestEvent  2026-01-01T12:00:01Z   

                                               actor  \
0  {'id': 250934060, 'login': 'dasepsamsul', 'dis...   
1  {'id': 41898282, 'login': 'github-actions[bot]...   
2  {'id': 104595665, 'login': 'aws-aemilia-kix', ...   
3  {'id': 176961590, 'login': 'gemini-code-assist...   
4  {'id': 112882, 'login': 'usta', 'display_login...   

                                                repo  \
0  {'id': 1120900126, 'name': 'dasepsamsul/Data-A...   
1  {'id': 1038951185, 'name': 'gschubert05/arb-ar...   
2  {'id': 1053834576, 'name': 'aws-aemilia-kix/Gi...   
3  {'id': 899737840, 'name': 'kyo1941/working_tim...   
4  {'id': 188828946, 'name': 'endo

In [11]:

# Display a breakdown of event types in this batch
print("\n--- Event Types Breakdown ---")
print(df["type"].value_counts())


--- Event Types Breakdown ---
type
PushEvent                        115695
PullRequestEvent                  11989
CreateEvent                        9118
IssueCommentEvent                  3619
WatchEvent                         3607
DeleteEvent                        3505
IssuesEvent                        2288
PullRequestReviewCommentEvent      1668
PullRequestReviewEvent             1533
ForkEvent                           666
ReleaseEvent                        593
MemberEvent                         188
CommitCommentEvent                   93
GollumEvent                          79
PublicEvent                          58
DiscussionEvent                      23
Name: count, dtype: int64


In [10]:
df_actor = pd.DataFrame(df["actor"].tolist())
df_repo = pd.DataFrame(df["repo"].tolist())
df_payload = pd.DataFrame(df["payload"].tolist())

In [11]:
df_actor.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             154722 non-null  int64
 1   login          154722 non-null  str  
 2   display_login  154722 non-null  str  
 3   gravatar_id    154722 non-null  str  
 4   url            154722 non-null  str  
 5   avatar_url     154722 non-null  str  
dtypes: int64(1), str(5)
memory usage: 7.1 MB


In [12]:
df_repo.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   id      154719 non-null  float64
 1   name    154719 non-null  str    
 2   url     154719 non-null  str    
dtypes: float64(1), str(2)
memory usage: 3.5 MB


In [13]:
df_payload.info()

<class 'pandas.DataFrame'>
RangeIndex: 154722 entries, 0 to 154721
Data columns (total 25 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   repository_id  115695 non-null  float64
 1   push_id        115695 non-null  float64
 2   ref            128318 non-null  str    
 3   head           115695 non-null  str    
 4   before         115695 non-null  str    
 5   ref_type       12623 non-null   str    
 6   full_ref       12623 non-null   str    
 7   pusher_type    12623 non-null   str    
 8   action         26267 non-null   str    
 9   issue          5907 non-null    object 
 10  comment        5380 non-null    object 
 11  number         11989 non-null   float64
 12  pull_request   15190 non-null   object 
 13  master_branch  9118 non-null    str    
 14  description    4139 non-null    str    
 15  label          5876 non-null    object 
 16  labels         5919 non-null    object 
 17  member         188 non-null     object 


In [19]:
df_payload["issue"]

0                                                       NaN
1                                                       NaN
2                                                       NaN
3         {'url': 'https://api.github.com/repos/kyo1941/...
4                                                       NaN
                                ...                        
154717                                                  NaN
154718                                                  NaN
154719                                                  NaN
154720                                                  NaN
154721                                                  NaN
Name: issue, Length: 154722, dtype: object

In [20]:
df_payload_issue = pd.DataFrame(df_payload["issue"].dropna().tolist())
df_payload_pull_request = pd.DataFrame(df_payload["pull_request"].dropna().tolist())
df_payload_comment = pd.DataFrame(df_payload["comment"].dropna().tolist())
df_payload_label = pd.DataFrame(df_payload["label"].dropna().tolist())
df_payload_labels = pd.DataFrame(df_payload["labels"].dropna().tolist())
df_payload_member = pd.DataFrame(df_payload["member"].dropna().tolist())
df_payload_release = pd.DataFrame(df_payload["release"].dropna().tolist())
df_payload_forkee = pd.DataFrame(df_payload["forkee"].dropna().tolist())
df_payload_review = pd.DataFrame(df_payload["review"].dropna().tolist())
df_payload_pages = pd.DataFrame(df_payload["pages"].dropna().tolist())
df_payload_assignee = pd.DataFrame(df_payload["assignee"].dropna().tolist())
df_payload_assignees = pd.DataFrame(df_payload["assignees"].dropna().tolist())
df_payload_discussion = pd.DataFrame(df_payload["discussion"].dropna().tolist())

In [21]:
df_payload_issue.info()

<class 'pandas.DataFrame'>
RangeIndex: 5907 entries, 0 to 5906
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   url                         5907 non-null   str   
 1   repository_url              5907 non-null   str   
 2   labels_url                  5907 non-null   str   
 3   comments_url                5907 non-null   str   
 4   events_url                  5907 non-null   str   
 5   html_url                    5907 non-null   str   
 6   id                          5907 non-null   int64 
 7   node_id                     5907 non-null   str   
 8   number                      5907 non-null   int64 
 9   title                       5907 non-null   str   
 10  user                        5907 non-null   object
 11  labels                      5907 non-null   object
 12  state                       5907 non-null   str   
 13  locked                      5907 non-null   bool  
 14  ass

In [22]:
df_payload_assignee.info()

<class 'pandas.DataFrame'>
RangeIndex: 555 entries, 0 to 554
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                555 non-null    str  
 1   id                   555 non-null    int64
 2   node_id              555 non-null    str  
 3   avatar_url           555 non-null    str  
 4   gravatar_id          555 non-null    str  
 5   url                  555 non-null    str  
 6   html_url             555 non-null    str  
 7   followers_url        555 non-null    str  
 8   following_url        555 non-null    str  
 9   gists_url            555 non-null    str  
 10  starred_url          555 non-null    str  
 11  subscriptions_url    555 non-null    str  
 12  organizations_url    555 non-null    str  
 13  repos_url            555 non-null    str  
 14  events_url           555 non-null    str  
 15  received_events_url  555 non-null    str  
 16  type                 555 non-null    

In [23]:
df_payload_assignees.info()

<class 'pandas.DataFrame'>
RangeIndex: 559 entries, 0 to 558
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       555 non-null    object
 1   1       270 non-null    object
 2   2       17 non-null     object
 3   3       14 non-null     object
 4   4       5 non-null      object
dtypes: object(5)
memory usage: 22.0+ KB


In [24]:
df_payload_comment.info()

<class 'pandas.DataFrame'>
RangeIndex: 5380 entries, 0 to 5379
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   url                       5380 non-null   str    
 1   html_url                  5380 non-null   str    
 2   issue_url                 3619 non-null   str    
 3   id                        5380 non-null   int64  
 4   node_id                   5380 non-null   str    
 5   user                      5380 non-null   object 
 6   created_at                5380 non-null   str    
 7   updated_at                5380 non-null   str    
 8   body                      5380 non-null   str    
 9   reactions                 5380 non-null   object 
 10  performed_via_github_app  2082 non-null   object 
 11  position                  1685 non-null   float64
 12  line                      0 non-null      float64
 13  path                      1685 non-null   str    
 14  commit_id          

In [25]:
df_payload_discussion.info()

<class 'pandas.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   repository_url      23 non-null     str   
 1   category            23 non-null     object
 2   answer_html_url     0 non-null      object
 3   answer_chosen_at    0 non-null      object
 4   answer_chosen_by    0 non-null      object
 5   html_url            23 non-null     str   
 6   id                  23 non-null     int64 
 7   node_id             23 non-null     str   
 8   number              23 non-null     int64 
 9   title               23 non-null     str   
 10  user                23 non-null     object
 11  labels              23 non-null     object
 12  state               23 non-null     str   
 13  state_reason        0 non-null      object
 14  locked              23 non-null     bool  
 15  comments            23 non-null     int64 
 16  created_at          23 non-null     str

In [26]:
df_payload_forkee.info()

<class 'pandas.DataFrame'>
RangeIndex: 666 entries, 0 to 665
Data columns (total 79 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           666 non-null    int64 
 1   node_id                      666 non-null    str   
 2   name                         666 non-null    str   
 3   full_name                    666 non-null    str   
 4   private                      666 non-null    bool  
 5   owner                        666 non-null    object
 6   html_url                     666 non-null    str   
 7   description                  543 non-null    str   
 8   fork                         666 non-null    bool  
 9   url                          666 non-null    str   
 10  forks_url                    666 non-null    str   
 11  keys_url                     666 non-null    str   
 12  collaborators_url            666 non-null    str   
 13  teams_url                    666 non-null    s

In [27]:
df_payload_label.info()

<class 'pandas.DataFrame'>
RangeIndex: 5876 entries, 0 to 5875
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           5876 non-null   int64
 1   node_id      5876 non-null   str  
 2   url          5876 non-null   str  
 3   name         5876 non-null   str  
 4   color        5876 non-null   str  
 5   default      5876 non-null   bool 
 6   description  4851 non-null   str  
dtypes: bool(1), int64(1), str(5)
memory usage: 281.3 KB


In [28]:
df_payload_labels.info()

<class 'pandas.DataFrame'>
RangeIndex: 5919 entries, 0 to 5918
Data columns (total 40 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       5876 non-null   object
 1   1       4826 non-null   object
 2   2       627 non-null    object
 3   3       337 non-null    object
 4   4       173 non-null    object
 5   5       75 non-null     object
 6   6       32 non-null     object
 7   7       28 non-null     object
 8   8       17 non-null     object
 9   9       13 non-null     object
 10  10      6 non-null      object
 11  11      6 non-null      object
 12  12      6 non-null      object
 13  13      6 non-null      object
 14  14      6 non-null      object
 15  15      6 non-null      object
 16  16      6 non-null      object
 17  17      6 non-null      object
 18  18      6 non-null      object
 19  19      6 non-null      object
 20  20      6 non-null      object
 21  21      6 non-null      object
 22  22      6 non-null      object
 23 

In [29]:
df_payload_member.info()

<class 'pandas.DataFrame'>
RangeIndex: 188 entries, 0 to 187
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   login                188 non-null    str  
 1   id                   188 non-null    int64
 2   node_id              188 non-null    str  
 3   avatar_url           188 non-null    str  
 4   gravatar_id          188 non-null    str  
 5   url                  188 non-null    str  
 6   html_url             188 non-null    str  
 7   followers_url        188 non-null    str  
 8   following_url        188 non-null    str  
 9   gists_url            188 non-null    str  
 10  starred_url          188 non-null    str  
 11  subscriptions_url    188 non-null    str  
 12  organizations_url    188 non-null    str  
 13  repos_url            188 non-null    str  
 14  events_url           188 non-null    str  
 15  received_events_url  188 non-null    str  
 16  type                 188 non-null    

In [30]:
df_payload_pages.info()

<class 'pandas.DataFrame'>
RangeIndex: 79 entries, 0 to 78
Columns: 218 entries, 0 to 217
dtypes: object(218)
memory usage: 134.7+ KB


In [31]:
df_payload_pull_request.info()

<class 'pandas.DataFrame'>
RangeIndex: 15190 entries, 0 to 15189
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   url     15190 non-null  str   
 1   id      15190 non-null  int64 
 2   number  15190 non-null  int64 
 3   head    15190 non-null  object
 4   base    15190 non-null  object
dtypes: int64(2), object(2), str(1)
memory usage: 593.5+ KB


In [32]:
df_payload_release.info()

<class 'pandas.DataFrame'>
RangeIndex: 593 entries, 0 to 592
Data columns (total 25 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   url                                  593 non-null    str    
 1   assets_url                           593 non-null    str    
 2   upload_url                           593 non-null    str    
 3   html_url                             593 non-null    str    
 4   id                                   593 non-null    int64  
 5   author                               593 non-null    object 
 6   node_id                              593 non-null    str    
 7   tag_name                             593 non-null    str    
 8   target_commitish                     593 non-null    str    
 9   name                                 590 non-null    str    
 10  draft                                593 non-null    bool   
 11  immutable                            593 no

In [33]:
df_payload_review.info()

<class 'pandas.DataFrame'>
RangeIndex: 1533 entries, 0 to 1532
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                1533 non-null   int64 
 1   node_id           1533 non-null   str   
 2   user              1533 non-null   object
 3   body              832 non-null    str   
 4   commit_id         1533 non-null   str   
 5   submitted_at      1533 non-null   str   
 6   state             1533 non-null   str   
 7   html_url          1533 non-null   str   
 8   pull_request_url  1533 non-null   str   
 9   _links            1533 non-null   object
 10  updated_at        1533 non-null   str   
dtypes: int64(1), object(2), str(8)
memory usage: 131.9+ KB
